# RES-142 — B2 Qwen3 scoring (Colab 2026.04 / A100)

**Offline A100 variant. All code cells remain unexecuted. The upload package is hash-pinned at two independent layers and keeps the frozen 20-query / 1,000-candidate population.**

The package is disabled until a separately controlled runtime watchdog has been independently qualified. A cell timer and projected CU cost are not a spending cap or a connection-to-disconnection control. Obtain explicit shutdown authorization before any future paid connection.

The A100 variant changes only accelerator checks and operational gates. It preserves the G4 scientific scorer, Qwen3 model/tokenizer revisions, BF16, scoring template, frozen candidates, and RRF baseline.


## Before any future paid connection

This notebook targets one NVIDIA A100 with compute capability 8.0 and at least 75,000 MiB visible memory. It pins the same Colab runtime, Linux Python 3.12 environment, CUDA 12.9/cu129 packages, PyTorch, vLLM, BF16 scoring path, model and template as the qualified G4 bundle.

The current package is **NO-GO**: it refuses to run because no independent controller has demonstrated automatic runtime termination and zero-active-session verification. The in-cell timer begins only when a cell starts and cannot cover a browser hang or a stalled upload. A projected CU amount is not an enforced cap.

The failed attempt's existing Drive evidence is immutable. A future reviewed attempt would use the create-only directory `b2-attempt-20261021-a100-repair-94e06560a2534a2d99174badbbcf4062/`; the orchestrator refuses to reuse it. No Colab connection, upload, or GPU allocation was made for this repair.


In [ ]:
from __future__ import annotations

import hashlib
import json
import time
import traceback
from pathlib import Path

from google.colab import files

RUN_STARTED = time.monotonic()
SESSION_LIMIT_SECONDS = 25 * 60
WORK_LIMIT_SECONDS = 23 * 60

# Keep these gates closed until final source review and a fresh numeric G4 spending authorization.
RUNTIME_VERSION_CONFIRMED = ""
SETUP_AUTHORIZATION = ""
SHUTDOWN_AUTHORIZATION = ""
RATE_INSPECTION_AUTHORIZATION = ""
RATE_INSPECTION_GPU = "A100"
RATE_INSPECTION_MAX_CU = 0.0
OBSERVED_CU_PER_HOUR = 0.0
APPROVED_MAX_CU = 0.0
DRIVE_AUTHORIZATION = ""
DRIVE_DESTINATION_CONFIRMATION = ""
SESSION_WATCHDOG_QUALIFIED = False

EXPECTED_SHA256 = {
    "res142_b2_colab.py": "aab769f06240cf129c045b37e5db4bf6b358e8c94a5fd85d438d243ba6d2a3ab",
    "res142_b2_runner.py": "7beb9fbf3a89deb8c47b32d39d85e1cc6636863256501a689aa39aca173f252b",
    "res142_b1_support.py": "74178e28b9cb8ac6e1a893e3fa07607a80869564caa6191ae81fbe8dde81f99e",
    "b2_execution.py": "2dc3ddd0a5a8aa9044d4eea38106e3920b36f8b741caca00d110ae4383b2701d",
    "res142-colab.lock.txt": "8b360f4533eb616eac38469e450d3df1a516f155afba464929036eada7d80e16",
    "res142_qwen3_reranker.jinja": "bf7b104801a4e4ad6a147d732e2d5a837a36f708cdbd1b871b864979d3e10acd",
    "res142-b2-candidate-input-v3.zip": "c98d26a7aaf38f35b66203f0d27a73f618800218b1a90c4e322a02ab2c4100ac",
    "rights-confirmation-v3.json": "65980a1e19e6a01444193df55e98fd90a941b4c14b4c83efbd32db3564449d92",
}
REQUIRED_FILES = set(EXPECTED_SHA256)
NORMALIZE_TEXT_FILES = {
    "res142_b2_colab.py", "res142_b2_runner.py", "res142_b1_support.py",
    "b2_execution.py", "res142-colab.lock.txt", "res142_qwen3_reranker.jinja",
}
DRIVE_ROOT = "/content/drive/MyDrive/DynamisRAG/RES-142"


def require_positive_number(value: object, label: str) -> float:
    if isinstance(value, bool) or not isinstance(value, (int, float)) or value <= 0:
        raise PermissionError(f"{label} must be entered as a positive number.")
    return float(value)


try:
    if SHUTDOWN_AUTHORIZATION != "APPROVE AUTOMATIC SESSION SHUTDOWN":
        raise PermissionError("Explicit shutdown authorization is required before a paid runtime.")
    if SESSION_WATCHDOG_QUALIFIED is not True:
        raise PermissionError("A100 launch is blocked: no independently qualified session watchdog.")
    if RUNTIME_VERSION_CONFIRMED != "2026.04":
        raise PermissionError("In Colab, select runtime 2026.04 and confirm it here.")
    if SETUP_AUTHORIZATION != "APPROVE COLAB SETUP":
        raise PermissionError("A fresh APPROVE COLAB SETUP authorization is required.")
    if RATE_INSPECTION_AUTHORIZATION != "APPROVE COLAB RATE INSPECTION":
        raise PermissionError("A fresh G4 rate inspection authorization is required.")
    if RATE_INSPECTION_GPU != "A100":
        raise PermissionError("The rate inspection and scoring target must both be A100.")
    if DRIVE_AUTHORIZATION != "APPROVE COLAB DRIVE STORAGE":
        raise PermissionError("A fresh Google Drive storage authorization is required.")
    if DRIVE_DESTINATION_CONFIRMATION.rstrip("/") != DRIVE_ROOT:
        raise PermissionError(f"Confirm the Drive root exactly as {DRIVE_ROOT!r}.")
    rate_limit = require_positive_number(RATE_INSPECTION_MAX_CU, "Rate-inspection maximum CU")
    observed_rate = require_positive_number(OBSERVED_CU_PER_HOUR, "Observed CU/hour rate")
    approved_max = require_positive_number(APPROVED_MAX_CU, "Approved maximum CU")
    projected_cu = observed_rate * SESSION_LIMIT_SECONDS / 3600
    if projected_cu > approved_max:
        raise PermissionError("The 25-minute CU projection exceeds a fresh numeric approval.")
    print(json.dumps({
        "gpu": "A100",
        "session_limit_seconds": SESSION_LIMIT_SECONDS,
        "work_deadline_seconds": WORK_LIMIT_SECONDS,
        "observed_cu_per_hour": observed_rate,
        "projected_cu": projected_cu,
        "approved_max_cu": approved_max,
        "projection_is_platform_enforced": False,
    }, indent=2))

    uploaded = files.upload()
    if set(uploaded) != REQUIRED_FILES:
        raise ValueError(f"Select exactly the reviewed eight files: {sorted(REQUIRED_FILES)}.")
    input_dir = Path("/content/res142-b2-input")
    if input_dir.exists():
        raise RuntimeError("Refusing to reuse /content/res142-b2-input; start with a fresh runtime.")
    input_dir.mkdir()
    for name, data in uploaded.items():
        if name in NORMALIZE_TEXT_FILES:
            data = data.replace(b"\r\n", b"\n")
        observed = hashlib.sha256(data).hexdigest()
        if observed != EXPECTED_SHA256[name]:
            raise ValueError(f"Uploaded artifact SHA-256 mismatch: {name}")
        path = input_dir / name
        with path.open("xb") as output_file:
            output_file.write(data)

    runner_path = input_dir / "res142_b2_colab.py"
    namespace = {"__name__": "res142_b2_colab_uploaded", "__file__": str(runner_path)}
    source = runner_path.read_text(encoding="utf-8")
    exec(compile(source, str(runner_path), "exec"), namespace)
    result = namespace["main"](
        upload_dir=input_dir,
        approvals={
            "setup": SETUP_AUTHORIZATION,
            "rate_inspection": RATE_INSPECTION_AUTHORIZATION,
            "rate_inspection_gpu": RATE_INSPECTION_GPU,
            "rate_inspection_max_compute_units": rate_limit,
            "gpu": "A100",
            "observed_compute_units_per_hour": observed_rate,
            "approved_max_compute_units": approved_max,
            "drive": DRIVE_AUTHORIZATION,
            "drive_destination_confirmation": DRIVE_DESTINATION_CONFIRMATION,
            "colab_orchestrator_sha256": EXPECTED_SHA256["res142_b2_colab.py"],
        },
        run_started=RUN_STARTED,
        runtime_version_confirmed=RUNTIME_VERSION_CONFIRMED,
    )
    print(json.dumps(result, indent=2, sort_keys=True))
except BaseException as error:
    output_root = Path("/content/res142-b2-preflight")
    output_root.mkdir(parents=True, exist_ok=True)
    already_reported = any(Path("/content/res142-b2/output").glob("failure-*.json"))
    if not already_reported:
        failure_path = output_root / f"failure-{time.time_ns()}.json"
        failure_path.write_text(json.dumps({
            "stage": "notebook_preflight_or_execution",
            "error_type": type(error).__name__,
            "message": str(error)[:2000],
            "traceback": traceback.format_exc()[-12000:],
            "runtime_seconds": time.monotonic() - RUN_STARTED,
        }, indent=2, sort_keys=True), encoding="utf-8")
        try:
            files.download(str(failure_path))
        except Exception:
            pass
    raise

## Frozen scoring profile and stop controls

The reviewed B2 scorer remains pinned to Qwen/Qwen3-Reranker-0.6B revision and tokenizer revision `e61197ed45024b0ed8a2d74b80b4d909f1255473`, the exact Jinja template SHA-256, BF16, vLLM 0.31.0/cu129, and batches of 8. The A100 checks require compute capability 8.0, at least 75,000 MiB visible VRAM, a compatible CUDA 12.9 driver, PyTorch 2.13.0, and BF16 support.

The scoring algorithm, query/candidate order, candidate identities, and original RRF baseline are unchanged. A different GPU can produce small floating point score differences, so bitwise score identity is not claimed. Candidate inputs and ranking rules remain fixed.

The cell-level 23-minute work deadline is an in-runtime fail-safe only. It does not control the connection lifetime. This notebook remains blocked until an external connection-to-disconnection watchdog is independently qualified and armed before startup.


## Output and operator stop

The scorer persists each completed query and the final evidence under the unique attempt directory, with create-only writes and read-back hashes. It does not write to the previous attempt directory.

This offline package is not launch-ready. Do not connect or upload until the independent watchdog can terminate a paid session without further confirmation and a separate check can verify zero active sessions. Obtain explicit shutdown authorization before any future paid connection.
